# Central Management Console Back End
Back End Connectivity for Central Management Console

In [ ]:
kv_uri = 'https://kvfabricprodeus2rh.vault.azure.net/'
client_id_secret = 'fuam-spn-client-id'
tenant_id_secret = 'fuam-spn-tenant-id'
client_secret_name = 'fuam-spn-secret'

workspace_id = 'a046cf0f-8dca-4b61-b95e-7adf68fb4b0a'
dataset_id = '708da792-a344-4079-b205-61c587a51600'



In [1]:
import requests
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.keyvault.secrets import SecretClient
import os
import notebookutils
import pandas as pd

StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 3, Finished, Available, Finished)

In [12]:
def get_api_token_via_akv(kv_uri:str, client_id_secret:str, tenant_id_secret:str, client_secret_name:str)->str:
    """
    Function to retrieve an api token used to authenticate with Microsoft Fabric APIs

    kv_uri:str: The uri of the azure key vault
    client_id_secret:str: The name of the key used to store the value for the client id in the akv
    tenant_id_secret:str: The name of the key used to store the value for the tenant id in the akv
    client_secret_name:str: The name of the key used to store the value for the client secret in the akv

    """
    client_id = notebookutils.credentials.getSecret(kv_uri, client_id_secret)
    tenant_id = notebookutils.credentials.getSecret(kv_uri, tenant_id_secret)
    client_secret = notebookutils.credentials.getSecret(kv_uri, client_secret_name)

    credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    scope = 'https://analysis.windows.net/powerbi/api/.default'
    token = credential.get_token(scope).token

    return token

def get_dataset_refresh_info(workspace_id:str, dataset_id:str, api_token:str)->pd.DataFrame:
    """
    https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/get-refresh-history-in-group
    scopes required: Dataset.ReadWrite.All or Dataset.Read.All

    GET https://api.powerbi.com/v1.0/myorg/groups/{groupId}/datasets/{datasetId}/refreshes

    workspace_id:str: The Workspace ID where the semantic model/dataset resides
    dataset_id:str: The Dataset ID to get refresh info for
    api_token:str: The api token to authenticate with the API

    returns:
        refresh_history_pd_df:pd.DataFrame: DataFrame of the refresh history
    """
    url = f'https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes'

    headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
    }    

    response = requests.get(url, headers=headers)

    return pd.DataFrame(response.json()['value'])

def start_dataset_refresh(workspace_id:str, dataset_id:str, api_token:str):
    """
    https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/refresh-dataset-in-group
    scopes required: Dataset.ReadWrite.All

    POST https://api.powerbi.com/v1.0/myorg/groups/{groupId}/datasets/{datasetId}/refreshes

    workspace_id:str: The workspace ID where the semantic model/dataset resides
    dataset_id:str: The Dataset ID to refresh
    api_token:str: The api token used to authenticate with the API

    returns:
        pass
    """
    url = f'https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes'

    headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
    }    

    response = requests.post(url, headers=headers)

    if response.status_code >=200 and response.status_code <300:
        print(f'Dataset Refresh request to workspace id:{workspace_id} and dataset id:{dataset_id} sent successfully')

    return response

def cancel_dataset_refresh(workspace_id:str, dataset_id:str, refresh_id:str, api_token:str):
    """
    https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/cancel-refresh-in-group
    scopes required: Dataset.ReadWrite.All

    DELETE https://api.powerbi.com/v1.0/myorg/groups/{groupId}/datasets/{datasetId}/refreshes/{refreshId}

    workspace_id:str: The workspace ID where the semantic model/dataset resides
    dataset_id:str: The Dataset ID of the active refresh to be cancelled
    api_token:str: The api token used to authenticate with the API

    returns:
        pass
    """
    url = f'https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes/{refresh_id}'

    headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
    }    

    response = requests.delete(url, headers=headers)

    if response.status_code==409:
        print(f'Dataset Refresh already in a completed state; cannot cancel')

    return response

def get_all_connections(api_token:str):
    """
    https://learn.microsoft.com/en-us/rest/api/fabric/core/connections/list-connections?tabs=HTTP
    scopes: Connection.Read.All or Connection.ReadWrite.All

    GET https://api.fabric.microsoft.com/v1/connections


    """
    url = 'https://api.fabric.microsoft.com/v1/connections'

    headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
    }    

    response = requests.get(url, headers=headers)

    return response


StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 14, Finished, Available, Finished)

In [3]:
# get oauth token
token = get_api_token_via_akv(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)

StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 5, Finished, Available, Finished)

In [10]:
# Get Dataset/SM Refresh Info
dataset_refresh_history = get_dataset_refresh_info(workspace_id, dataset_id, token)

dataset_refresh_history

StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 12, Finished, Available, Finished)

,requestId,id,refreshType,startTime,endTime,status,refreshAttempts,extendedStatus
0,2b2abe5c-330e-436b-bcd9-8c099254bc4a,34484184,ViaApi,2025-06-17T20:12:37.823Z,2025-06-17T20:12:47.97Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T20:...",NaN
1,a7545f09-c85c-42df-91fe-de326b61ebe8,34484126,ViaApi,2025-06-17T20:10:12.213Z,2025-06-17T20:10:25.463Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T20:...",NaN
2,028eb9ac-fdbc-4b5b-ba7e-1c41720a83aa,34482527,ViaApi,2025-06-17T19:18:49.263Z,2025-06-17T19:18:56.817Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T19:...",NaN
3,8d537681-efd7-48d3-bff3-4a33fe0c03de,34482524,ViaApi,2025-06-17T19:18:24.417Z,2025-06-17T19:18:31.283Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T19:...",NaN
4,1f0a8650-f3ab-4e8c-888e-215364cfb080,34482500,ViaApi,2025-06-17T19:17:31.757Z,2025-06-17T19:17:40.113Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T19:...",NaN
5,f6516407-f639-4433-9428-92b8d034127e,34481659,ViaApi,2025-06-17T18:56:04.19Z,2025-06-17T18:56:11.81Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T18:...",NaN
6,2f0cf6bf-8671-48a2-adfe-73000ac82306,34481648,ViaApi,2025-06-17T18:55:31.337Z,2025-06-17T18:55:39.51Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T18:...",NaN
7,f6468215-17f3-4481-a4ca-5568f1ff889d,34481576,ViaApi,2025-06-17T18:53:38.113Z,2025-06-17T18:53:52.197Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T18:...",NaN
8,c1f802df-57a6-434a-bad2-75d73a1b9a23,34479286,ViaApi,2025-06-17T17:36:11.583Z,2025-06-17T17:36:23.62Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T17:...",NaN
9,75f6a10f-e11a-ce16-0ebc-1eaa2e6450c1,34479143,OnDemand,2025-06-17T17:30:33.607Z,2025-06-17T17:30:41.513Z,Completed,"[{'attemptId': 1, 'startTime': '2025-06-17T17:...",NaN


In [8]:
# Start Dataset/SM Refresh
# https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/refresh-dataset-in-group

resp = start_dataset_refresh(workspace_id, dataset_id, token)

StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 10, Finished, Available, Finished)

Dataset Refresh request to workspace id:a046cf0f-8dca-4b61-b95e-7adf68fb4b0a and dataset id:708da792-a344-4079-b205-61c587a51600 sent successfully


In [11]:
# Cancel Dataset/SM Refresh
# https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/cancel-refresh-in-group
refresh_id = '2b2abe5c-330e-436b-bcd9-8c099254bc4a'

cancel_resp = cancel_dataset_refresh(workspace_id, dataset_id, refresh_id, token)


StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 13, Finished, Available, Finished)

Dataset Refresh already in a completed state; cannot cancel


In [7]:
cancel_resp.status_code

StatementMeta(, c51fe2fc-4917-4a83-8492-684f51cc89bb, 9, Finished, Available, Finished)

409